# 18_rag: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/18_rag/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Build overlapping chunks while retaining an article's citation identity.

Run: uv run  python modules/18_rag/examples/01_from_scratch.py
Expected: overlapping windows share help-000#procedure.
CPU: seconds to one minute. Needs: base, help articles. No implicit downloads.
"""

from common.apps.rag import chunk_article
from common.settings import data_dir


def main() -> None:
    path = data_dir() / "help_center/help-000.md"
    # KEY IDEA: window IDs distinguish chunks, but citations refer to stable passages.
    windows = chunk_article(path, words=20, overlap=5)
    for chunk in windows:
        print(chunk.id, chunk.passage_id, chunk.text)
    assert len(windows) > 1
    assert len({c.passage_id for c in windows}) == 1


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/18_rag/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("qdrant_client") is None:
    print("SKIP native comparison: --extra qdrant")
else:
    """Index real help-center embeddings and retrieve scoped passages in Qdrant.

    Run: uv run --extra qdrant python modules/18_rag/examples/02_production.py
    Expected: buyer/north policy passages with stable source IDs.
    CPU: seconds to one minute. Needs: qdrant extra, prepared RAG vectors. No implicit downloads.
    """
    # course-extra: qdrant

    import time

    from common.apps.database import AppDatabase
    from common.pretty import show_hits
    from common.tier4 import open_rag

    def main() -> None:
        with AppDatabase("qdrant-local") as db:
            rag, questions, queries = open_rag(db)
            q = questions[0]
            start = time.perf_counter()
            # KEY IDEA: role and region are eligibility, not hints to the generator.
            hits = rag.retrieve(q["question"], queries[q["id"]])
            show_hits(hits, (time.perf_counter() - start) * 1000, title="Retrieved policy evidence")
            print([h.metadata["passage_id"] for h in hits])
            assert all(
                h.metadata["role"] == "buyer" and h.metadata["region"] == "north" for h in hits
            )

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/18_rag/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure how a narrow IVF probe and post-filtering lose RAG evidence.

Run: uv run  python modules/18_rag/examples/03_measure.py
Expected: exact and approximate passage recall over 80 questions.
CPU: seconds to one minute. Needs: base, frozen help vectors. No implicit downloads.
"""

from common.apps.database import AppDatabase
from common.apps.evaluation import rag_evaluation
from common.teaching import evidence


def main() -> None:
    with AppDatabase() as db:
        # KEY IDEA: assess source passage recall before asking the generator to answer.
        result = rag_evaluation(db)
    print({k: v for k, v in result.items() if k != "rows"})
    evidence("rag-index-quality", result, tier=4)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 04_generation

Source: `modules/18_rag/examples/04_generation.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("ollama") is None or "ollama" not in os.environ.get(
    "MARLOWE_NOTEBOOK_PROFILES", ""
).split(","):
    print(
        "SKIP native comparison: --extra rag; start ollama and include it in MARLOWE_NOTEBOOK_PROFILES"
    )
else:
    """Generate a cited answer using only retrieved policy evidence and local Ollama.

    Run: uv run --extra rag python modules/18_rag/examples/04_generation.py
    Expected: an answer plus citations and a separate unresolved faithfulness field.
    CPU: seconds to one minute. Needs: rag extra, local Ollama qwen3:0.6b, prepared help vectors. No implicit downloads.
    """
    # course-extra: rag
    # course-profile: ollama

    from common.apps.database import AppDatabase
    from common.teaching import evidence
    from common.tier4 import open_rag

    def main() -> None:
        with AppDatabase() as db:
            rag, questions, queries = open_rag(db)
            q = questions[0]
            hits = rag.retrieve(q["question"], queries[q["id"]])
            # KEY IDEA: a valid source ID does not establish that the claim is supported.
            result = rag.answer(q["question"], hits)
        print(result)
        evidence("rag-generation", result, tier=4)

    if __name__ == "__main__":
        from common.lesson_runtime import run_server_lesson

        run_server_lesson(main, "ollama")

## 05_abstention

Source: `modules/18_rag/examples/05_abstention.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Do not call a generator when there is no eligible evidence.

Run: uv run  python modules/18_rag/examples/05_abstention.py
Expected: abstained=True without a model call.
CPU: seconds to one minute. Needs: base, frozen help vectors. No implicit downloads.
"""

from common.apps.database import AppDatabase
from common.tier4 import open_rag


def main() -> None:
    with AppDatabase() as db:
        rag, _, _ = open_rag(db)

        def forbidden(_):
            raise AssertionError("An empty context must not call the generator")

        # KEY IDEA: abstention is an explicit application outcome.
        result = rag.answer("Will my investment double tomorrow?", [], generate=forbidden)
        assert result["abstained"] and result["generation_ms"] == 0
        print(result)


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.apps.rag import chunk_article
from common.settings import data_dir

chunks = chunk_article(data_dir() / "help_center/help-000.md", words=15, overlap=4)
print([(c.id, c.passage_id) for c in chunks])

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.